In [1]:
!pip install -q "mlflow[databricks]>=3.1"

In [2]:
import os
import mlflow
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["DATABRICKS_HOST"] = secrets.get_secret("DATABRICKS_HOST")
os.environ["DATABRICKS_TOKEN"] = secrets.get_secret("DATABRICKS_TOKEN")

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Users/sarmadmirza8888@gmail.com/my-ai-agent")

# Automatically trace every LangChain call (agent runs, tool calls, retriever calls, LLM calls)
mlflow.langchain.autolog()

print("Connected! MLflow autologging enabled for LangChain.")

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


Connected! MLflow autologging enabled for LangChain.


In [3]:
# Uninstall first to avoid version conflicts with the Kaggle base image
!pip uninstall -y -q transformers

!pip install -q "transformers==4.57.3" mistral-common accelerate bitsandbytes
print("Core model dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 36.2 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 6.1.0 requires huggingface-hub<2.0.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.
sentence-transformers 6.1.0 requires transformers<6.0.0,>=5.0.0, but you have transformers 4.57.3 which is incompatible.
Core model dependencies installed.


In [4]:
!pip install -q -U \
    langchain \
    langchain-core \
    langchain-classic \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    sentence-transformers \
    faiss-cpu \
    pypdf \
    ddgs
print("LangChain + retrieval dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 5.9 MB/s eta 0:00:00a 0:00:01
LangChain + retrieval dependencies installed.


In [5]:
pip install -U langchain-core langchain-classic langchain-community langchain-huggingface langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import glob
import json

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_classic.agents import AgentExecutor, create_react_agent

print("Imports successful!")

Imports successful!


In [7]:
def find_model_path():
    candidates = glob.glob("/kaggle/input/**/config.json", recursive=True)

    if not candidates:
        raise FileNotFoundError(
            "No model folder found under /kaggle/input. "
            "Attach the Mistral model dataset to this notebook."
        )

    mistral_candidates = [c for c in candidates if "mistral" in c.lower()]
    chosen = mistral_candidates[0] if mistral_candidates else candidates[0]

    return os.path.dirname(chosen)


MODEL_PATH = find_model_path()
print("Using model folder:", MODEL_PATH)

Using model folder: /kaggle/input/datasets/sarmadmirza12/mistral-7b-instruct-4bit


In [8]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

print("Loading Mistral model (this can take a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto"
)

print("Mistral model loaded successfully!")

Loading tokenizer...
Loading Mistral model (this can take a few minutes)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Mistral model loaded successfully!


In [9]:
def ask_mistral(prompt, max_new_tokens=200):
    """Quick sanity-check helper: raw prompt in, raw text out."""
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": prompt},
    ]

    # return_dict=True gives back a proper BatchEncoding with input_ids +
    # attention_mask, which we unpack explicitly with **. Relying on
    # return_tensors="pt" alone is ambiguous across transformers versions
    # (it can hand back a raw tensor OR a dict-like object depending on the
    # tokenizer's chat template), which is what caused the "shape" KeyError.
    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )

    input_length = encoded["input_ids"].shape[-1]
    generated_tokens = outputs[0][input_length:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


print(ask_mistral("Explain what a vector database is in simple words."))

A vector database is a type of database that stores and manages data in the form of vectors, which are mathematical representations of data points as ordered lists of numbers. In simpler terms, it's like a database that deals with data in a more complex, multi-dimensional format compared to traditional databases that work with simpler data types like numbers and text.

Vectors are used in various fields such as machine learning, computer vision, and natural language processing. They are used to represent complex data like images, audio, and text. For example, an image can be converted into a vector of numbers that represent its pixel values.

The main advantage of a vector database is that it allows for efficient searching and analysis of similar vectors. This is particularly useful in applications like image recognition, recommendation systems, and anomaly detection. Instead of searching for exact matches, a vector database can find items that are similar to a given query, even if the

In [10]:
json_files = glob.glob("/kaggle/input/**/*.json", recursive=True)
# Don't accidentally pick up the model's own config/tokenizer json files
json_files = [f for f in json_files if os.path.dirname(f) != os.path.dirname(MODEL_PATH + "/x")]

print("Number of CV JSON files found:", len(json_files))
for f in json_files[:10]:
    print(" -", f)

Number of CV JSON files found: 500
 - /kaggle/input/datasets/sarmadmirza12/zipcvv/ZIP/cv (478)_annotated.json
 - /kaggle/input/datasets/sarmadmirza12/zipcvv/ZIP/cv (6)_annotated.json
 - /kaggle/input/datasets/sarmadmirza12/zipcvv/ZIP/cv (204)_annotated.json
 - /kaggle/input/datasets/sarmadmirza12/zipcvv/ZIP/cv (330)_annotated.json
 - /kaggle/input/datasets/sarmadmirza12/zipcvv/ZIP/cv (488)_annotated.json
 - /kaggle/input/datasets/sarmadmirza12/zipcvv/ZIP/cv (46)_annotated.json
 - /kaggle/input/datasets/sarmadmirza12/zipcvv/ZIP/cv (268)_annotated.json
 - /kaggle/input/datasets/sarmadmirza12/zipcvv/ZIP/cv (413)_annotated.json
 - /kaggle/input/datasets/sarmadmirza12/zipcvv/ZIP/cv (222)_annotated.json
 - /kaggle/input/datasets/sarmadmirza12/zipcvv/ZIP/cv (381)_annotated.json


In [11]:
documents = []

for json_path in json_files:
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        text = data.get("text", "")
        if not text:
            continue

        documents.append(
            Document(page_content=text, metadata={"source": json_path})
        )

    except Exception as e:
        print("Error reading:", json_path, "->", e)

print("Total CV documents loaded:", len(documents))

Total CV documents loaded: 495


In [12]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print("Total CV documents:", len(documents))
print("Total chunks:", len(chunks))

Total CV documents: 495
Total chunks: 2575


In [13]:
print("Loading embedding model...")
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)
print("Embedding model loaded successfully!")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [14]:
print("Creating FAISS vector store...")
vectorstore = FAISS.from_documents(chunks, embedding_model)
print("FAISS vector store created successfully!")

Creating FAISS vector store...
FAISS vector store created successfully!


In [15]:
def search_cvs(query: str) -> str:
    retrieved_docs = vectorstore.similarity_search(query, k=3)

    results = []
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "Unknown")
        results.append(f"Source: {source}\nContent:\n{doc.page_content}")

    return "\n\n---\n\n".join(results)


# Quick sanity check
print(search_cvs("Who has experience with Python and Docker?"))

Source: /kaggle/input/datasets/sarmadmirza12/zipcvv/ZIP/cv (83)_annotated.json
Content:
authorized desktop applications, hardware, and peripheral equipment. 
 Coordinate and execute preventative maintenance and remedial repairs on 
computers, laptops, printers, and peripherals. 
 Monitor, operate, manage, troubleshoot, and restore service to terminal 
service clients, PCs, or laptops.  
 
KEY SKILLS AND COMPETENCIES  
 
 Good understanding of Server Hardware Technology, UNIX, 
WINDOWS, SQL, LANs, WANs.  
 
 Experience in VMware, Citrix, Windows, Linux, Active Directory 
Group Policies, TCP/IP, DNS, HDCP, NAT, and Apache HTTP 
Server. 
 
 Programming Skills:  C/C++/Java, HTML, CSS, MySQL, and 
MATLAB. 
 
 Network and System Security (Hands on Experience):  Manual SQL 
Injection, No redirect Exploit, DDOS attack, Shell Upload (XSS), 
Nessus Vulnerability Scanner, Acunetix Security Testing Tool. 
 
 Hands of experience on Backtrack Linux. Install and configure 
various switches, ro

In [16]:
def clean_react_output(text: str) -> str:
    """
    The local Mistral model sometimes keeps generating after it has already
    given the real answer, and starts hallucinating a brand-new
    Question/Thought/Human turn on its own (visible in earlier runs as
    stray "Question:" text tacked onto the end of an answer, e.g.
    "Final Answer: 100\n\nQuestion:").

    If that leaked, half-formed turn gets returned as a tool's output, the
    calling agent treats it as real context and runs off on an unrelated,
    invented task (this is the root cause of the hallucination where the
    main agent starts answering questions nobody asked). This trims any
    output back to just the real answer by cutting at the first sign the
    model started a new turn by itself.
    """
    if not text:
        return text

    cut_markers = ["\nQuestion:", "\nThought:", "\nHuman:", "\nAI:", "\nObservation:"]
    for marker in cut_markers:
        idx = text.find(marker)
        if idx != -1:
            text = text[:idx]

    return text.strip()


In [17]:
from langchain_classic.agents.output_parsers.react_single_input import (
    ReActSingleInputOutputParser,
    FINAL_ANSWER_ACTION,
)
import re


class GroundedReActOutputParser(ReActSingleInputOutputParser):
    """
    The local model sometimes gets impatient: instead of stopping right
    after "Action Input: ...", it also bolts on a guessed Final Answer in
    the same turn (e.g. "Final Answer: ... are [candidate names]" -- note
    the unfilled placeholder, it never actually looked anything up).

    The stock parser treats "Action + Final Answer in the same turn" as a
    hard error. That error text then gets stitched into the *next* prompt
    as if it were part of the transcript, and the model hallucinates
    further trying to make sense of it (e.g. inventing a fake
    `visit_documentation` tool out of the troubleshooting link in the
    error message).

    Fix: when both appear, trust the Action and silently drop the guessed
    Final Answer, so the real tool actually gets called and the eventual
    answer is grounded in a real Observation instead of a guess.
    """

    def parse(self, text: str):
        if FINAL_ANSWER_ACTION in text:
            action_regex = (
                r"Action\s*\d*\s*:[\s]*(.*?)"
                r"Action\s*\d*\s*Input\s*\d*\s*:[\s]*(.*)"
            )
            if re.search(action_regex, text, re.DOTALL):
                text = text.split(FINAL_ANSWER_ACTION)[0].strip()
        return super().parse(text)


grounded_parser = GroundedReActOutputParser()


In [18]:
@tool
def addition_tool(numbers: str) -> str:
    """Add two numbers provided as comma-separated values."""

    numbers = numbers.replace("Observation", "").strip()

    try:
        a, b = map(float, numbers.split(","))
        return str(a + b)
    except Exception:
        return "Error: provide two numbers separated by a comma."
@tool
def subtraction_tool(numbers: str) -> str:
    """Subtract the second number from the first number using two comma-separated numbers."""

    numbers = numbers.replace("Observation", "").strip()

    try:
        a, b = map(float, numbers.split(","))
        return str(a - b)
    except Exception:
        return "Error: provide two numbers separated by a comma."


@tool
def multiplication_tool(numbers: str) -> str:
    """Multiply two numbers provided as comma-separated values."""

    numbers = numbers.replace("Observation", "").strip()

    try:
        a, b = map(float, numbers.split(","))
        return str(a * b)
    except Exception:
        return "Error: provide two numbers separated by a comma."


@tool
def division_tool(numbers: str) -> str:
    """Divide the first number by the second number using two comma-separated numbers."""

    numbers = numbers.replace("Observation", "").strip()

    try:
        a, b = map(float, numbers.split(","))

        if b == 0:
            return "Error: cannot divide by zero."

        return str(a / b)

    except Exception:
        return "Error: provide two numbers separated by a comma."

In [19]:
text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,          # was 400 - a shorter cap so a runaway generation can't fabricate multiple fake Q&A turns
    do_sample=False,
    return_full_text=False,
    stop_strings=["\nObservation", "\nQuestion:", "\nThought:", "\nHuman:"],  # stop the instant the model tries to open a NEW turn (fabricated Question/Thought/Human), not just the two cases we originally guarded against
)
text_generation_pipeline._forward_params["tokenizer"] = tokenizer


llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

print("Local LLM wrapper created successfully.")

Local LLM wrapper created successfully.


In [20]:
calculator_tools = [
    addition_tool,
    subtraction_tool,
    multiplication_tool,
    division_tool
]
CALCULATOR_PROMPT_TEMPLATE = """You are a calculator specialist.

Your job is to solve mathematical calculations accurately.

You have access to the following calculator tools:

{tools}

Use the following format exactly:

Question: the calculation you need to solve
Thought: think about which calculator operation is required
Action: the action to take, must be exactly one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action

Thought: I now know the final answer
Final Answer: the final answer to the original question

Only handle mathematical calculations.

Question: {input}
Thought:{agent_scratchpad}"""
calculator_prompt = PromptTemplate.from_template(
    CALCULATOR_PROMPT_TEMPLATE
)
calculator_agent = create_react_agent(
    llm,
    calculator_tools,
    calculator_prompt,
    output_parser=grounded_parser,
)
calculator_executor = AgentExecutor(
    agent=calculator_agent,
    tools=calculator_tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=4,
    early_stopping_method="force",
)
def run_calculator_agent(user_query):
    result = calculator_executor.invoke({
        "input": user_query
    })

    return clean_react_output(result["output"])
print(run_calculator_agent("What is 25 multiplied by 4?"))



> Entering new AgentExecutor chain...
I need to multiply two numbers
Action: multiplication_tool
Action Input: 25, 4
Observation100.025 multiplied by 4 is 100
Final Answer: 100

Question:

> Finished chain.
100


In [21]:
print(addition_tool.invoke("2,2"))

4.0


In [22]:
print(addition_tool.args_schema.model_json_schema())

{'description': 'Add two numbers provided as comma-separated values.', 'properties': {'numbers': {'title': 'Numbers', 'type': 'string'}}, 'required': ['numbers'], 'title': 'addition_tool', 'type': 'object'}


In [23]:
@tool
def search_cvs_tool(query: str) -> str:
    """Search the CV database for candidates, skills, education, experience, and qualifications."""
    return search_cvs(query)

In [24]:
cv_tools = [
    search_cvs_tool
]
CV_AGENT_PROMPT_TEMPLATE = """You are a CV search specialist.

Your job is to answer questions about candidates and their
skills, experience, education, projects, and other information
contained in the CV database.

You have access to the following tool:

{tools}

Use the following format exactly:

Question: the input question you must answer
Thought: you should think about what information is needed
Action: the action to take, must be exactly one of [{tool_names}]
Action Input: the search query for the CV database
Observation: the result returned by the CV search tool

Thought: I now know the final answer
Final Answer: the final answer to the original question

Important rules:
- Use the CV search tool for CV-related questions.
- Base your answer on the retrieved CV information..
- Do not invent candidate information.

Question: {input}
Thought:{agent_scratchpad}"""
cv_prompt = PromptTemplate.from_template(
    CV_AGENT_PROMPT_TEMPLATE
)
cv_agent = create_react_agent(
    llm,
    cv_tools,
    cv_prompt,
    output_parser=grounded_parser,
)
cv_executor = AgentExecutor(
    agent=cv_agent,
    tools=cv_tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=4,
    early_stopping_method="force",
)
def run_cv_agent(user_query):
    result = cv_executor.invoke({
        "input": user_query
    })

    return clean_react_output(result["output"])


In [25]:
print(
    run_cv_agent(
        "Who has experience with Python and Docker?"
    )
)



> Entering new AgentExecutor chain...
I need to find candidates with experience in both Python and Docker.
Action: search_cvs_tool
Action Input: "Python AND Docker"
ObservationSource: /kaggle/input/datasets/sarmadmirza12/zipcvv/ZIP/cv (459)_annotated.json
Content:
for all users
Maintain Inventory Management System for Desktop Computers, Servers, Generating reports/list for
audit purpose and records.
Provide resources to Project, mainly Computer & Printers as per project requirement while handling
installation of required software, OS, Anti-virus and Network connectivity. 
Analysis of customers LAN/WAN s to determine network performance problems. Provided solutions,
and documentation.
Plan, design, install, configure, upgrade and maintain LAN, as per organizational requirements,
governed by communication protocols. 
Manage computer installations along with all related accessories and networking to ensure hassle
free  functioning  of  systems.  Updated  users  on maintenance  tips

In [26]:
web_search_tool = DuckDuckGoSearchRun()
web_search_tool.description = (
    "Search the internet for general, current, recent, or external information "
    "that is not related to the CV database."
)
web_tools = [
    web_search_tool
]
WEB_AGENT_PROMPT_TEMPLATE = """You are a web search specialist.

Your job is to answer questions that require information from
the internet.

You have access to the following tool:

{tools}

Use the following format exactly:

Question: the input question you must answer
Thought: think about what information is needed
Action: the action to take, must be exactly one of [{tool_names}]
Action Input: the search query for the internet
Observation: the result returned by the web search tool

Thought: I now know the final answer
Final Answer: the final answer to the original question

Important rules:
- Use the web search tool for internet-related questions.
- Search the internet when current or external information is required.
- Base your answer on the information returned by the web search tool.
- Do not invent information.

Question: {input}
Thought:{agent_scratchpad}"""
web_prompt = PromptTemplate.from_template(
    WEB_AGENT_PROMPT_TEMPLATE
)

web_agent = create_react_agent(
    llm,
    web_tools,
    web_prompt,
    output_parser=grounded_parser,
)
web_executor = AgentExecutor(
    agent=web_agent,
    tools=web_tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=4,
    early_stopping_method="force",
)
def run_web_agent(user_query):
    result = web_executor.invoke({
        "input": user_query
    })

    return clean_react_output(result["output"])


In [27]:
print(
    run_web_agent(
        "What is the latest version of Python?"
    )
)



> Entering new AgentExecutor chain...
I need to find the latest version of Python
Action: duckduckgo_search
Action Input: latest version of Python
ObservationPython 3.15.0 is the newest major release of the Python programming language, and it contains many new features and optimisations compared to Python 3.14, in 5,643 commits from 1,012 contributors. The official home of the Python Programming Language Security content in all five releases Additional security content by version Python 3.10.22, 3.11.17, 3.12.15 and 3.13.16 Python 3.13.16 and 3.14.8 Python 3.10.22 Attention macOS 27.0 IDLE or tkinter users And now for something completely different We hope you enjoy the new releases! Status of Python versions ¶ The main branch is currently the future Python 3.16, and is the only branch that accepts new features. The latest release for each Python version can be found on the download page. Python 3.15.0 is the newest major release of the Python programming language, and it contains ma

In [28]:
@tool
def cv_search_agent_tool(query: str) -> str:
    """Use this subagent to search the CV database for candidate skills, experience, education, projects, and other CV information."""
    
    return run_cv_agent(query)
@tool
def calculator_agent_tool(query: str) -> str:
    """Use this subagent for mathematical calculations and arithmetic operations."""
    
    return run_calculator_agent(query)
@tool
def web_search_agent_tool(query: str) -> str:
    """Use this subagent to search the internet for current, recent, or external information."""
    
    return run_web_agent(query)
    

In [29]:
main_tool= [
    cv_search_agent_tool,
    calculator_agent_tool,
    web_search_agent_tool
]

In [30]:


test_output = text_generation_pipeline(
    "Question: Who knows Python?\nThought: I should search.\nAction: search_cvs_tool\nAction Input: Python\nObservation:"
)
print("----- stop_strings sanity check -----")
print(test_output[0]["generated_text"])
print("--------------------------------------")

----- stop_strings sanity check -----
Python is a popular programming language.
Thought:
--------------------------------------


In [31]:
MAIN_AGENT_PROMPT_TEMPLATE = """You are the main coordinator agent.

Your ONLY job is to coordinate the specialist agents.

You have access to the following specialist agents:

{tools}

Your process is:

1. Understand the user's original request.
2. Decide which specialist agent or agents are needed.
3. Delegate the request to the appropriate specialist agent.
4. Receive the specialist's result.
5. Return the specialist's result as the final answer when it is sufficient.

Use the following format exactly:

Question: the user's original request
Thought: decide which specialist agent is needed
Action: the specialist agent to call, must be exactly one of [{tool_names}]
Action Input: the complete original request to send to the specialist agent
Observation: the result returned by the specialist agent

Thought: the specialist has provided the result needed for the original request
Final Answer: the answer to the original user request

Important rules:

- You are ONLY a coordinator.
- Do not perform calculations yourself.
- Do not search the CV database yourself.
- Do not search the internet yourself.
- Do not invent tool names.
- Do not write Python code as an Action.
- Do not create a new Question.
- Do not invent a new user request.
- Do not continue with an unrelated task.
- After receiving a sufficient specialist result, immediately give the Final Answer.
- Do not call another specialist after receiving a sufficient result.
- Only call another specialist if the ORIGINAL user request genuinely requires additional information.

Previous conversation (may be empty):
{chat_history}

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

react_prompt = PromptTemplate.from_template(MAIN_AGENT_PROMPT_TEMPLATE)

main_agent = create_react_agent(llm, main_tool, react_prompt, output_parser=grounded_parser)

main_executor = AgentExecutor(
    agent=main_agent,
    tools=main_tool,
    verbose=True,             # prints each Thought/Action/Observation - great for debugging
    handle_parsing_errors=True,  # if the model's output doesn't match the format, feed the error back instead of crashing
    max_iterations=2,         # cap so a confused small model can't loop forever
    early_stopping_method="force",  # "generate" is not a valid option in this langchain-classic version; "force" just returns whatever answer it has when max_iterations is hit
)

print("ReAct agent created successfully.")

ReAct agent created successfully.


In [32]:
def run_agent(user_query, chat_history_text=""):
    result = main_executor.invoke({
        "input": user_query,
        "chat_history": chat_history_text,
    })

    answer = clean_react_output(result["output"])

    updated_history = (
        chat_history_text
        + f"\nHuman: {user_query}\nAI: {answer}"
    )

    return answer, updated_history

In [33]:
# ============================================================
#  PART 1a - The summariser
#  Compresses old conversation turns into a short factual summary.
# ============================================================

SUMMARY_PROMPT = """You are maintaining a running summary of a conversation
between a Human and an AI assistant.

Summary of the conversation so far:
{existing_summary}

New lines of conversation to fold into that summary:
{new_lines}

Write an UPDATED summary in 100 words or less.
Keep concrete facts: candidate names, skills, numbers, results, and what the
user is trying to achieve. Drop greetings and filler.
Output only the summary text - no preamble, no bullet points, no new questions.

Updated summary:"""


def summarize_text(existing_summary: str, new_lines: str, max_new_tokens: int = 180) -> str:
    """Call the local Mistral model directly (NOT through the ReAct pipeline).

    We bypass `text_generation_pipeline` on purpose: that pipeline is configured
    with ReAct stop strings and a 200-token cap for agent reasoning. Summarising
    is a plain completion task, so we want deterministic decoding instead.
    """
    prompt = SUMMARY_PROMPT.format(
        existing_summary=existing_summary.strip() or "(nothing yet)",
        new_lines=new_lines.strip(),
    )

    messages = [
        {"role": "system", "content": "You write short, factual conversation summaries."},
        {"role": "user", "content": prompt},
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,      # a summary should be faithful, not creative
        )

    input_length = encoded["input_ids"].shape[-1]
    text = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()

    # Reuse the same guard as the agents: a small model sometimes tries to
    # open a brand-new turn after finishing the summary.
    return clean_react_output(text)


print("Summariser ready.")

Summariser ready.


In [34]:
# ============================================================
#  PART 1b - Summary-buffer memory  (now serialisable)
# ============================================================

class ConversationSummaryBufferMemory:
    """Recent turns kept verbatim, older turns compressed into a summary.

    Parameters
    ----------
    keep_last_n : how many recent turns stay word-for-word in the window.
    summarize_every : how many turns must overflow the window before we pay
        for a summarisation call (batching keeps it cheap).
    max_answer_chars : long answers get clipped inside the window so one huge
        CV dump cannot eat the whole context.
    """

    def __init__(self, keep_last_n=4, summarize_every=2, max_answer_chars=400):
        self.keep_last_n = keep_last_n
        self.summarize_every = summarize_every
        self.max_answer_chars = max_answer_chars

        self.summary = ""        # compressed older history
        self.buffer = []         # [(question, answer), ...] recent, verbatim
        self.pending = []        # turns that fell out, awaiting summarisation
        self.all_turns = []      # full log, never trimmed (for inspection only)

    # -- internal helpers ------------------------------------------------

    def _clip(self, text):
        text = (text or "").strip()
        if len(text) > self.max_answer_chars:
            return text[: self.max_answer_chars].rstrip() + " ..."
        return text

    def _flush_pending(self):
        """Fold the overflowed turns into the running summary."""
        if not self.pending:
            return

        new_lines = "\n".join(
            f"Human: {q}\nAI: {self._clip(a)}" for q, a in self.pending
        )
        self.summary = summarize_text(self.summary, new_lines)
        self.pending = []

    def add_turn(self, question, answer):
        self.all_turns.append((question, answer))
        self.buffer.append((question, answer))

        # Anything past the window falls into `pending`.
        while len(self.buffer) > self.keep_last_n:
            self.pending.append(self.buffer.pop(0))

        if len(self.pending) >= self.summarize_every:
            self._flush_pending()

    def get_context(self) -> str:
        """The string that gets injected into {chat_history} in the prompt."""
        parts = []

        if self.summary:
            parts.append(f"Summary of earlier conversation:\n{self.summary}")

        if self.buffer:
            recent = "\n".join(
                f"Human: {q}\nAI: {self._clip(a)}" for q, a in self.buffer
            )
            parts.append(f"Most recent turns:\n{recent}")

        return "\n\n".join(parts)

    def clear(self):
        self.summary = ""
        self.buffer = []
        self.pending = []
        self.all_turns = []

    def show(self):
        print("=" * 60)
        print(f"Turns stored : {len(self.all_turns)}")
        print(f"In window    : {len(self.buffer)} (verbatim)")
        print(f"Awaiting sum.: {len(self.pending)}")
        print("-" * 60)
        print("SUMMARY:", self.summary or "(empty)")
        print("=" * 60)

    # -- serialisation ---------------------------------------------------
    #  Only the summary + the current window are worth saving. `all_turns`
    #  is a debug log; persisting it would grow the file forever for no gain.

    def state_dict(self):
        return {
            "summary": self.summary,
            "buffer": [list(t) for t in self.buffer],
            "pending": [list(t) for t in self.pending],
        }

    def load_state_dict(self, state):
        if not state:
            return
        self.summary = state.get("summary", "") or ""
        self.buffer = [tuple(t) for t in state.get("buffer", []) if len(t) == 2]
        self.pending = [tuple(t) for t in state.get("pending", []) if len(t) == 2]
        self.all_turns = list(self.pending) + list(self.buffer)


print("ConversationSummaryBufferMemory ready.")

ConversationSummaryBufferMemory ready.


In [35]:
# ============================================================
#  PART 2 - Semantic history cache
#  "If the user already asked this, answer from history."
# ============================================================

import re
import time
import numpy as np
from datetime import datetime

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


def normalize_question(q: str) -> str:
    """Lowercase, strip punctuation, collapse whitespace.

    This is what makes "What is 2 + 5?" and "what is 2+5" hash to the
    same key in the exact-match layer.
    """
    q = q.lower()
    q = re.sub(r"[^a-z0-9\s]", " ", q)
    q = re.sub(r"\s+", " ", q)
    return q.strip()   # strip LAST - punctuation becomes a space, so "2+5?" would
                       # otherwise leave a trailing space and never match "2 + 5"


# Questions whose correct answer changes over time must NOT be cached forever.
TIME_SENSITIVE_WORDS = {
    "latest", "current", "currently", "today", "now", "recent", "recently",
    "news", "price", "weather", "version", "newest", "update", "updated",
    "yesterday", "tomorrow", "live",
}


class SemanticHistoryCache:
    """Two-layer question lookup over past (question, answer) pairs.

    Layer 1 - exact match on the normalised string: O(1) dict lookup.
    Layer 2 - cosine similarity over sentence embeddings: O(n) dot product.

    Both vectors are L2-normalised, so the dot product IS the cosine
    similarity (cos = a.b / (|a||b|), and |a| = |b| = 1).

    Each entry holds: question, answer, vector, timestamp.
    """

    def __init__(self, embedder, threshold=0.90, ttl_seconds=None):
        self.embedder = embedder
        self.threshold = threshold      # tune this - see the test cell below
        self.ttl_seconds = ttl_seconds  # None = entries never expire

        self.questions = []
        self.answers = []
        self.vectors = []               # list of normalised np.float32 arrays
        self.timestamps = []
        self.exact_index = {}           # normalised question -> position

        self._matrix = None             # cached np.vstack of self.vectors
        self.hits = 0
        self.misses = 0
        self.loaded_from_disk = 0

    # -- internal helpers ------------------------------------------------

    def _embed(self, text):
        v = np.asarray(self.embedder.embed_query(text), dtype="float32")
        return self._unit(v)

    @staticmethod
    def _unit(v):
        norm = np.linalg.norm(v)
        return v / norm if norm > 0 else v

    def _get_matrix(self):
        """Stack vectors once and reuse. Rebuilt only when an entry is added."""
        if self._matrix is None and self.vectors:
            self._matrix = np.vstack(self.vectors)
        return self._matrix

    def _is_fresh(self, i):
        if self.ttl_seconds is None:
            return True
        return (time.time() - self.timestamps[i]) < self.ttl_seconds

    def is_cacheable(self, question: str) -> bool:
        tokens = set(normalize_question(question).split())
        return not (tokens & TIME_SENSITIVE_WORDS)

    # -- public API ------------------------------------------------------

    def add(self, question, answer, vector=None, timestamp=None):
        if not self.is_cacheable(question):
            return False  # e.g. "what is the latest version of Python"

        key = normalize_question(question)

        if key in self.exact_index:          # already seen: refresh in place
            i = self.exact_index[key]
            self.answers[i] = answer
            self.timestamps[i] = timestamp or time.time()
            return False

        self.exact_index[key] = len(self.questions)
        self.questions.append(question)
        self.answers.append(answer)
        self.vectors.append(self._embed(question) if vector is None else self._unit(vector))
        self.timestamps.append(timestamp or time.time())
        self._matrix = None               # invalidate the cached stack
        return True

    def lookup(self, question):
        """Return a hit dict, or None if the agent has to do real work."""
        if not self.questions or not self.is_cacheable(question):
            self.misses += 1
            return None

    
        key = normalize_question(question)
        if key in self.exact_index:
            i = self.exact_index[key]
            if self._is_fresh(i):
                self.hits += 1
                return {
                    "answer": self.answers[i],
                    "score": 1.0,
                    "matched_question": self.questions[i],
                    "method": "exact",
                    "age_seconds": time.time() - self.timestamps[i],
                }

      
        query_vec = self._embed(question)
        sims = self._get_matrix() @ query_vec   # cosine sim for every entry

        best = int(np.argmax(sims))
        best_score = float(sims[best])

        if best_score >= self.threshold and self._is_fresh(best):
            self.hits += 1
            return {
                "answer": self.answers[best],
                "score": best_score,
                "matched_question": self.questions[best],
                "method": "semantic",
                "age_seconds": time.time() - self.timestamps[best],
            }

        self.misses += 1
        return None

    def rank(self, question, top_k=3):
        """Debug helper: show the closest stored questions and their scores."""
        if not self.questions:
            print("(cache is empty)")
            return

        sims = self._get_matrix() @ self._embed(question)
        order = np.argsort(sims)[::-1][:top_k]

        print(f"Query: {question}")
        for i in order:
            mark = "HIT " if sims[i] >= self.threshold else "miss"
            print(f"  [{mark}] {sims[i]:.3f}  <- {self.questions[i]}")

    def stats(self):
        total = self.hits + self.misses
        rate = (self.hits / total * 100) if total else 0
        print(f"Cached questions: {len(self.questions)} "
              f"({self.loaded_from_disk} loaded from disk) | "
              f"hits: {self.hits} | misses: {self.misses} | hit rate: {rate:.1f}%")

    def clear(self):
        self.questions, self.answers = [], []
        self.vectors, self.timestamps = [], []
        self.exact_index = {}
        self._matrix = None
        self.hits = self.misses = self.loaded_from_disk = 0

    # -- serialisation ---------------------------------------------------

    def export_entries(self):
        """question / answer / vector / timestamp, ready for JSON."""
        out = []
        for q, a, v, ts in zip(self.questions, self.answers,
                               self.vectors, self.timestamps):
            out.append({
                "question": q,
                "normalized": normalize_question(q),
                "answer": a,
                # 6 decimals is far below what cosine similarity can resolve
                # and roughly halves the file size vs full float repr
                "vector": [round(float(x), 6) for x in v],
                "timestamp": ts,
                "saved_at": datetime.fromtimestamp(ts).isoformat(timespec="seconds"),
            })
        return out

    def import_entries(self, entries, expected_dim=None, re_embed=False):
        """Load entries back in. Returns (loaded, skipped).

        re_embed=True recomputes vectors from the stored question text, which
        is what we do if the embedding model changed since the file was written.
        """
        loaded = skipped = 0

        for e in entries:
            try:
                q = e["question"]
                a = e["answer"]
                ts = float(e.get("timestamp") or time.time())
                vec = None

                if not re_embed:
                    raw = e.get("vector")
                    if not raw:
                        skipped += 1
                        continue
                    vec = np.asarray(raw, dtype="float32")
                    if expected_dim and vec.shape[0] != expected_dim:
                        skipped += 1
                        continue
                    # rounding perturbed the norm slightly - renormalise so the
                    # dot product is still exactly the cosine similarity
                    vec = self._unit(vec)

                if self.add(q, a, vector=vec, timestamp=ts):
                    loaded += 1
                else:
                    skipped += 1

            except (KeyError, TypeError, ValueError):
                skipped += 1

        self.loaded_from_disk += loaded
        return loaded, skipped


print("SemanticHistoryCache ready.")

SemanticHistoryCache ready.


In [36]:
# ============================================================
#  PART 3 - Persistence:  history.json
# ============================================================
#  Schema
#  ------
#  {
#    "schema_version": 1,
#    "embedding_model": "...",      # so we can detect a model change
#    "vector_dim": 384,
#    "created_at": <unix ts>,
#    "updated_at": <unix ts>,
#    "memory":  {"summary": "...", "buffer": [...], "pending": [...]},
#    "entries": [ {question, normalized, answer, vector, timestamp, saved_at} ]
#  }

import os
import glob
import shutil

SCHEMA_VERSION = 1


def resolve_history_path():
    """Write to /kaggle/working when on Kaggle, else the current folder."""
    if os.path.isdir("/kaggle/working"):
        return "/kaggle/working/history.json"
    return os.path.abspath("history.json")


HISTORY_PATH = resolve_history_path()


def find_seed_history():
    """First run on a fresh session: look for a history.json we saved earlier.

    If you attached a previous notebook's output as a dataset, it shows up
    under /kaggle/input (read-only), so we copy it into the working dir.
    """
    if os.path.exists(HISTORY_PATH):
        return None

    candidates = glob.glob("/kaggle/input/**/history.json", recursive=True)
    return candidates[0] if candidates else None


def save_history(memory, cache, path=HISTORY_PATH, quiet=False):
    """Write the whole history atomically.

    Atomic = write to a temp file, then os.replace(). If the kernel dies
    mid-write you keep the last good file instead of a truncated one.
    """
    created = time.time()
    if os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8") as f:
                created = json.load(f).get("created_at", created)
        except Exception:
            pass

    payload = {
        "schema_version": SCHEMA_VERSION,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "vector_dim": int(cache.vectors[0].shape[0]) if cache.vectors else None,
        "created_at": created,
        "updated_at": time.time(),
        "memory": memory.state_dict(),
        "entries": cache.export_entries(),
    }

    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)
    os.replace(tmp, path)

    if not quiet:
        size_kb = os.path.getsize(path) / 1024
        print(f"[history] saved {len(payload['entries'])} entries "
              f"-> {path} ({size_kb:.1f} KB)")

    return path


def load_history(memory, cache, path=HISTORY_PATH, verbose=True):
    """Restore memory + cache from disk. Safe to call when no file exists."""

    seed = find_seed_history()
    if seed:
        shutil.copy(seed, path)
        if verbose:
            print(f"[history] seeded from attached dataset: {seed}")

    if not os.path.exists(path):
        if verbose:
            print(f"[history] no file yet - one will be created at {path}")
        return False

    # --- read, tolerating a corrupt file ---
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except (json.JSONDecodeError, OSError) as e:
        backup = f"{path}.corrupt-{int(time.time())}"
        shutil.move(path, backup)
        print(f"[history] file unreadable ({e}); moved to {backup}. Starting fresh.")
        return False

    # --- does the saved model match the one currently loaded? ---
    saved_model = data.get("embedding_model")
    re_embed = bool(saved_model and saved_model != EMBEDDING_MODEL_NAME)
    if re_embed:
        print(f"[history] embedding model changed "
              f"({saved_model} -> {EMBEDDING_MODEL_NAME}); re-embedding questions.")

    memory.load_state_dict(data.get("memory"))
    loaded, skipped = cache.import_entries(
        data.get("entries", []),
        expected_dim=data.get("vector_dim"),
        re_embed=re_embed,
    )

    if verbose:
        age_days = (time.time() - data.get("created_at", time.time())) / 86400
        print(f"[history] loaded {loaded} cached answers from {path} "
              f"(skipped {skipped}, file age {age_days:.1f} days)")
        if memory.summary:
            print(f"[history] restored summary: {memory.summary[:120]}...")

    return True


print(f"Persistence ready. History file: {HISTORY_PATH}")

Persistence ready. History file: /kaggle/working/history.json


In [37]:
# ============================================================
#  PART 4 - Wire memory + cache + persistence into the agent
# ============================================================

memory = ConversationSummaryBufferMemory(keep_last_n=4, summarize_every=2)
qa_cache = SemanticHistoryCache(embedding_model, threshold=0.90)

# Creates history.json on the first run, reuses it on every run after that.
load_history(memory, qa_cache)


def run_agent_with_memory(user_query, use_cache=True, show_trace=True, persist=True):
    """Replacement for run_agent().

      1. Ask the history cache first - if we already answered this, return it.
      2. Otherwise build the context window (summary + recent turns).
      3. Run the coordinator agent.
      4. Record the turn in memory + cache, then write history.json.
    """
    # ---- Step 1: can history answer this already? ----
    if use_cache:
        hit = qa_cache.lookup(user_query)
        if hit:
            if show_trace:
                age_h = hit["age_seconds"] / 3600
                print(f"[history] answered from memory "
                      f"({hit['method']}, similarity={hit['score']:.3f}, "
                      f"stored {age_h:.1f}h ago)")
                print(f"[history] matched earlier question: {hit['matched_question']}")
            memory.add_turn(user_query, hit["answer"])
            if persist:
                save_history(memory, qa_cache, quiet=True)
            return hit["answer"]

    # ---- Step 2: build the bounded context window ----
    context = memory.get_context()

    # ---- Step 3: real agent run ----
    result = main_executor.invoke({
        "input": user_query,
        "chat_history": context,
    })
    answer = clean_react_output(result["output"])

    # ---- Step 4: remember it, on disk as well as in RAM ----
    memory.add_turn(user_query, answer)
    qa_cache.add(user_query, answer)
    if persist:
        save_history(memory, qa_cache, quiet=True)

    return answer


print("run_agent_with_memory ready.")
qa_cache.stats()

[history] loaded 16 cached answers from /kaggle/working/history.json (skipped 0, file age 29.7 days)
[history] restored summary: The user seeks Python-skilled individuals for a project, inquires about machine learning, and requests suitable candidat...
run_agent_with_memory ready.
Cached questions: 16 (16 loaded from disk) | hits: 0 | misses: 0 | hit rate: 0.0%


In [38]:

pairs = [

    ("Who has experience with Python and Docker?",
     "Which candidates know Python and Docker?", True),
    ("What is 25 multiplied by 4?",
     "what is 25 * 4",                            True),
    ("Who has experience with Python?",
     "Who has experience with Java?",             False),
    ("Who has a Master's degree?",
     "How many candidates are there?",            False),
]

probe = SemanticHistoryCache(embedding_model, threshold=0.90)

print(f"{'score':>7}  {'same?':>6}  {'fires?':>7}  question pair")
print("-" * 78)
for a, b, same in pairs:
    va, vb = probe._embed(a), probe._embed(b)
    score = float(va @ vb)
    fires = score >= probe.threshold
    flag = "  <-- CHECK" if fires != same else ""
    print(f"{score:>7.3f}  {str(same):>6}  {str(fires):>7}  {a[:34]!r} / {b[:34]!r}{flag}")

  score   same?   fires?  question pair
------------------------------------------------------------------------------
  0.773    True    False  'Who has experience with Python and' / 'Which candidates know Python and D'  <-- CHECK
  0.827    True    False  'What is 25 multiplied by 4?' / 'what is 25 * 4'  <-- CHECK
  0.528   False    False  'Who has experience with Python?' / 'Who has experience with Java?'
  0.154   False    False  "Who has a Master's degree?" / 'How many candidates are there?'


In [39]:
# ============================================================
#  PART 4 - Wire memory + cache + persistence into the agent
# ============================================================

memory = ConversationSummaryBufferMemory(keep_last_n=4, summarize_every=2)
qa_cache = SemanticHistoryCache(embedding_model, threshold=0.90)

# Creates history.json on the first run, reuses it on every run after that.
load_history(memory, qa_cache)


@mlflow.trace(name="cv_agent_query")
def run_agent_with_memory(user_query, use_cache=True, show_trace=True, persist=True):
    """Replacement for run_agent().

      1. Ask the history cache first - if we already answered this, return it.
      2. Otherwise build the context window (summary + recent turns).
      3. Run the coordinator agent.
      4. Record the turn in memory + cache, then write history.json.
    """
    # ---- Step 1: can history answer this already? ----
    if use_cache:
        hit = qa_cache.lookup(user_query)
        if hit:
            if show_trace:
                age_h = hit["age_seconds"] / 3600
                print(f"[history] answered from memory "
                      f"({hit['method']}, similarity={hit['score']:.3f}, "
                      f"stored {age_h:.1f}h ago)")
                print(f"[history] matched earlier question: {hit['matched_question']}")
            memory.add_turn(user_query, hit["answer"])
            if persist:
                save_history(memory, qa_cache, quiet=True)
            mlflow.update_current_trace(
                tags={"answer_source": "cache", "cache_method": hit["method"]}
            )
            return hit["answer"]

    # ---- Step 2: build the bounded context window ----
    context = memory.get_context()

    # ---- Step 3: real agent run ----
    result = main_executor.invoke({
        "input": user_query,
        "chat_history": context,
    })
    answer = clean_react_output(result["output"])

    # ---- Step 4: remember it, on disk as well as in RAM ----
    memory.add_turn(user_query, answer)
    qa_cache.add(user_query, answer)
    if persist:
        save_history(memory, qa_cache, quiet=True)

    mlflow.update_current_trace(tags={"answer_source": "agent"})
    return answer


print("run_agent_with_memory ready.")
qa_cache.stats()

[history] loaded 16 cached answers from /kaggle/working/history.json (skipped 0, file age 29.7 days)
[history] restored summary: The user seeks Python-skilled individuals for a project, inquires about machine learning, and requests suitable candidat...
run_agent_with_memory ready.
Cached questions: 16 (16 loaded from disk) | hits: 0 | misses: 0 | hit rate: 0.0%


In [40]:
import time

questions = [
    "What is 25 multiplied by 4?",   # miss  -> full agent run
    "what is 25 multiplied by 4",    # exact match (normalised)
    "Calculate 25 times 4",          # semantic match
]

for q in questions:
    start = time.time()
    answer = run_agent_with_memory(q, show_trace=True)
    elapsed = time.time() - start
    print(f"Q: {q}")
    print(f"A: {answer}")
    print(f"   ({elapsed:.2f}s)")
    print("-" * 60)

qa_cache.stats()

[history] answered from memory (exact, similarity=1.000, stored 713.8h ago)
[history] matched earlier question: What is 25 multiplied by 4?
Q: What is 25 multiplied by 4?
A: 100
   (0.02s)
------------------------------------------------------------
[history] answered from memory (exact, similarity=1.000, stored 713.8h ago)
[history] matched earlier question: What is 25 multiplied by 4?
Q: what is 25 multiplied by 4
A: 100
   (13.03s)
------------------------------------------------------------
[history] answered from memory (exact, similarity=1.000, stored 713.8h ago)
[history] matched earlier question: Calculate 25 times 4
Q: Calculate 25 times 4
A: 100 is the result of 25 times 4
   (0.01s)
------------------------------------------------------------
Cached questions: 16 (16 loaded from disk) | hits: 3 | misses: 0 | hit rate: 100.0%


In [41]:
import os, json

if os.path.exists(HISTORY_PATH):
    with open(HISTORY_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"File        : {HISTORY_PATH}")
    print(f"Size        : {os.path.getsize(HISTORY_PATH) / 1024:.1f} KB")
    print(f"Schema      : v{data['schema_version']}")
    print(f"Embedder    : {data['embedding_model']} (dim={data['vector_dim']})")
    print(f"Entries     : {len(data['entries'])}")
    print(f"Summary     : {data['memory']['summary'][:100] or '(empty)'}")
    print("-" * 60)

    for e in data["entries"][:3]:
        print(f"question  : {e['question']}")
        print(f"answer    : {e['answer'][:70]}")
        print(f"timestamp : {e['timestamp']:.0f}  ({e['saved_at']})")
        print(f"vector    : dim={len(e['vector'])}  "
              f"first 5 = {[round(x, 4) for x in e['vector'][:5]]}")
        print("-" * 60)
else:
    print("No history file yet - run a question through run_agent_with_memory first.")

File        : /kaggle/working/history.json
Size        : 67.3 KB
Schema      : v1
Embedder    : sentence-transformers/all-MiniLM-L6-v2 (dim=384)
Entries     : 16
Summary     : The user seeks Python-skilled individuals for their project, inquiring about Python and Docker. The 
------------------------------------------------------------
question  : What is 25 multiplied by 4?
answer    : 100
timestamp : 1789663509  (2026-09-17T16:45:08)
vector    : dim=384  first 5 = [0.0571, 0.0301, -0.0437, -0.0427, -0.0973]
------------------------------------------------------------
question  : Calculate 25 times 4
answer    : 100 is the result of 25 times 4
timestamp : 1789663520  (2026-09-17T16:45:19)
vector    : dim=384  first 5 = [0.0661, 0.0688, 0.0058, -0.1026, -0.0904]
------------------------------------------------------------
question  : Who has experience with Kubernetes?
answer    : Ali Raza lists Kubernetes.
timestamp : 1790331406  (2026-09-25T10:16:45)
vector    : dim=384  first 5 = [0

In [42]:
# Write something to history first (no agent call - we insert it directly)
qa_cache.add("Who has experience with Kubernetes?", "Ali Raza lists Kubernetes.")
memory.add_turn("Who has experience with Kubernetes?", "Ali Raza lists Kubernetes.")
save_history(memory, qa_cache)

print()
print("--- simulating a kernel restart: discarding everything in RAM ---")
del memory, qa_cache

# Exactly what the wiring cell does on a fresh session
memory = ConversationSummaryBufferMemory(keep_last_n=4, summarize_every=2)
qa_cache = SemanticHistoryCache(embedding_model, threshold=0.90)
load_history(memory, qa_cache)

print()
hit = qa_cache.lookup("Which candidate knows Kubernetes?")   # reworded on purpose
if hit:
    print(f"RECOVERED via {hit['method']} match (score {hit['score']:.3f})")
    print("Answer:", hit["answer"])
else:
    print("Not recovered - check the threshold with qa_cache.rank(...)")
    qa_cache.rank("Which candidate knows Kubernetes?")

[history] saved 16 entries -> /kaggle/working/history.json (67.2 KB)

--- simulating a kernel restart: discarding everything in RAM ---
[history] loaded 16 cached answers from /kaggle/working/history.json (skipped 0, file age 29.7 days)
[history] restored summary: The user is seeking Python-skilled individuals for their project, with requirements including Python, Docker, and machin...

Not recovered - check the threshold with qa_cache.rank(...)
Query: Which candidate knows Kubernetes?
  [miss] 0.613  <- Who has experience with Kubernetes?
  [miss] 0.445  <- find the candidates who knows machine learning
  [miss] 0.437  <- find the candidates who knows python and docker?


In [43]:
print("=" * 60)
print("       CV ANALYZER AGENT  (summary memory + persistent history)")
print("Ask about CVs, calculations, or general information.")
print("Commands: memory | cache | save | reset | exit")
print(f"History file: {HISTORY_PATH}")
print("=" * 60)
qa_cache.stats()

while True:
    user_query = input("\nYou: ").strip()

    if not user_query:
        continue

    low = user_query.lower()

    if low in ["exit", "quit"]:
        save_history(memory, qa_cache)
        print("Agent stopped.")
        break

    if low == "memory":
        memory.show()
        continue

    if low == "cache":
        qa_cache.stats()
        continue

    if low == "save":
        save_history(memory, qa_cache)
        continue

    if low == "reset":
        memory.clear()
        qa_cache.clear()
        if os.path.exists(HISTORY_PATH):
            os.remove(HISTORY_PATH)
        print("Memory, cache and history.json cleared.")
        continue

    try:
        answer = run_agent_with_memory(user_query)
        print("\nAgent:", answer)
    except Exception as e:
        print("Error:", e)

       CV ANALYZER AGENT  (summary memory + persistent history)
Ask about CVs, calculations, or general information.
Commands: memory | cache | save | reset | exit
History file: /kaggle/working/history.json
Cached questions: 16 (16 loaded from disk) | hits: 0 | misses: 1 | hit rate: 0.0%



You:  find those who know java




> Entering new AgentExecutor chain...
I need to search the CV database for candidates with Java skills.
Action: cv_search_agent_tool
Action Input: "Java"
Observation

> Entering new AgentExecutor chain...
I need to find candidates with Java skills
Action: search_cvs_tool
Action Input: "skills: Java"
ObservationSource: /kaggle/input/datasets/sarmadmirza12/zipcvv/ZIP/cv (490)_annotated.json
Content:
Technical  Skills :
Theoretical Knowledge Practical Knowledge
Data Structure, DBMS, Operating 
System, Networking.C, JAVA, HTML, PHP, SQL, VISUAL 
STUDIO, Photoshop.
Training :
Summer training in “ Advanced Java” at the GLOBSYN Finishing School 
from 11th July 2016 to 
                 10th August 2016. 
Winter training in “ Mobile Application in Android” at the WEBTECH LABS 
PVT. LTD. from 25th  
                 December 2015  to 31st January 2016. 
Project :
Advanced  Java Project (J2EE) on Online Hardware and Software Support
System.
Database Management System using C# in Visual Studi

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


3 candidates with Java skills have been found
Final Answer: 3 candidates with Java skills have been found.

> Finished chain.
3 candidates with Java skills have been found.3 candidates with Java skills have been found.
Final Answer: Here are the 3 candidates with Java skills: [Candidate 1, Candidate 2, Candidate 3]

> Finished chain.

Agent: Here are the 3 candidates with Java skills: [Candidate 1, Candidate 2, Candidate 3]


...



> Entering new AgentExecutor chain...
I need to search the internet for the history of the user's previous requests
Action: web_search_agent_tool
Action Input: show history
Observation

> Entering new AgentExecutor chain...
I don't have access to a user's history, so I can't show it.
Final Answer: I'm sorry, I don't have access to a user's history.

Question:

> Finished chain.
I'm sorry, I don't have access to a user's history.I have provided the result needed for the original request
Final Answer: I don't have access to a user's history.

> Finished chain.

Agent: I don't have access to a user's history.



You:  what we have discussed so far




> Entering new AgentExecutor chain...
the user is still seeking Python-skilled individuals for their project, requiring Python, Docker, machine learning, and preferably Kubernetes. John Doe, Jane Doe, and Ali Raza are proficient in Python, with John Doe and Jane Doe also having experience with Docker and machine learning. Ali Raza also has experience with Kubernetes.
Final Answer: We have discussed the project requirements, which are Python, Docker, machine learning, and preferably Kubernetes. The candidates proficient in these skills are John Doe, Jane Doe, and Ali Raza. John Doe and Jane Doe have experience with Docker and machine learning, while Ali Raza has experience with Kubernetes.

> Finished chain.

Agent: We have discussed the project requirements, which are Python, Docker, machine learning, and preferably Kubernetes. The candidates proficient in these skills are John Doe, Jane Doe, and Ali Raza. John Doe and Jane Doe have experience with Docker and machine learning, while 


You:  exit


[history] saved 21 entries -> /kaggle/working/history.json (88.7 KB)
Agent stopped.
